## Model Evaluation

### Imports

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import find_peaks, medfilt
from scipy.interpolate import make_interp_spline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import  KFold, GroupKFold

from model_functions import *
from helper_functions import *

In [ ]:
CSV_FEATURES = r"PATH_TO_CSV.csv"
CSV_METADATA = r"PATH_TO_METADATA.csv"

features, amplitude, speed, trial_names, trial_numbers, handedness, motor_skills = load_speed_amp(CSV_FEATURES, CSV_METADATA, None, exclude_ids=[6, 16])

ids = features["ids"].values
n_participants = len(np.unique(ids))
n_trials = len(features)

print(f"=== Data Overview ===")
print(f"Participants after exclusion: {n_participants}")
print(f"Trials after exclusion: {n_trials} \n")
print(f"=== Average Cycle Duration (Speed) ===")
print(f"Mean: {np.mean(speed):.4f} s")
print(f"Std: {np.std(speed):.4f} s")
print(f"Min: {np.min(speed):.4f} s")
print(f"Max: {np.max(speed):.4f} s \n")
print(f"=== Average Amplitude ===")
print(f"Mean: {np.mean(amplitude):.4f} °")
print(f"Std: {np.std(amplitude):.4f} °")
print(f"Min: {np.min(amplitude):.4f} °")
print(f"Max: {np.max(amplitude):.4f} ° \n")
corr, p = spearmanr(speed, amplitude)
print(f"=== Global Spearman Correlation (Speed vs Amplitude) ===")
print(f"Spearman r: {corr:.4f}")
print(f"p-value: {p:.4f}")

### Holdout Split

In [ ]:
models = {"Linear": (fit_linear_model, predict_linear_model),
          "Exponential": (fit_exponential_model, predict_exponential_model),
          "Logarithmic": (fit_logarithmic_model, predict_logarithmic_model),
          "Logistic": (fit_logistic_model, predict_logistic_model)}

groups = features["ids"].values

speed_train, amp_train, ids_train, speed_holdout, amp_holdout, ids_holdout = holdout_split(
    speed, amplitude, groups, n_holdout=5)

pareto_speed_train, pareto_amp_train, pareto_ids_train = pool_pareto_points(
    speed_train, amp_train, ids_train)

pareto_speed_holdout, pareto_amp_holdout, pareto_ids_holdout = pool_pareto_points(
    speed_holdout, amp_holdout, ids_holdout)

group_kf = GroupKFold(n_splits=5)

### GroupKFold (Global) on Pareto-Points-Pooled Data

In [ ]:
global_results_pareto = {}

print("=== Global GroupKFold Cross Validation on Pareto-Points-Pooled Data ===\n")
for model_name, (fit_function, predict_function) in models.items():
    metrics = evaluate_model_cv(fit_function, predict_function, pareto_speed_train, pareto_amp_train, group_kf, groups=pareto_ids_train)
    global_results_pareto[model_name] = metrics
    print_metrics(model_name, metrics)    

### GroupKFold (Global) on Full Train-/Test-Data

In [ ]:
global_results_full = {}

print("=== Global GroupKFold Cross Validation on Full Train-/Test-Data ===\n")
for model_name, (fit_function, predict_function) in models.items():
    metrics = evaluate_model_cv(fit_function, predict_function, speed_train, amp_train, group_kf, groups=ids_train)
    global_results_full[model_name] = metrics
    print_metrics(model_name, metrics)

### Per-Participant KFold

In [ ]:
print("=== Per-Participant KFold Cross-Validation ===\n")
per_participant_results = {model_name: [] for model_name in models}

ids = features["ids"].values  
for pid in np.unique(ids):
    pid_mask = ids == pid
    speed_p = speed[pid_mask]
    amp_p = amplitude[pid_mask]

    if len(speed_p) < 5:
        print(f"Skipping participant {pid}: not enough trials for 5-fold CV")
        continue

    print(f"============= PARTICIPANT {pid} ")
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for model_name, (fit_function, predict_function) in models.items():
        metrics = evaluate_model_cv(fit_function, predict_function, speed_p, amp_p, kf)
        per_participant_results[model_name].append(metrics)
        print_metrics(model_name, metrics)

### Aggregate Per-Participant Results

In [ ]:
print("=== Per-Participant Cross-Validation: Aggregated Across Participants ===\n")
for model_name, results_list in per_participant_results.items():
    agg_rmse = np.mean([r["RMSE"][0] for r in results_list])
    agg_mae = np.mean([r["MAE"][0] for r in results_list])
    agg_r2 = np.mean([r["R2"][0] for r in results_list])
    agg_spearman = np.mean([r["Spearman"][0] for r in results_list])
    
    print(f"====== {model_name}")
    print(f"RMSE:     {agg_rmse:.4f}")
    print(f"MAE:      {agg_mae:.4f}")
    print(f"R^2:      {agg_r2:.4f}")
    print(f"Spearman: {agg_spearman:.4f}\n")

### Best Model Selection and Final Evaluation on Pareto-Pooled Data

In [ ]:
best_model_name = None
best_r2 = -np.inf

for model_name, metrics in global_results_pareto.items():
    r2 = metrics["R2"][0]
    if r2 > best_r2:
        best_r2 = r2
        best_model_name = model_name

print(f"Best model: {best_model_name} (R^2 = {best_r2:.4f}) \n")

fit_function, predict_function = models[best_model_name]
final_model = fit_function(pareto_speed_train, pareto_amp_train)
holdout_pred = predict_function(final_model, pareto_speed_holdout)

print(f"=== Final Holdout Evaluation on Pareto-Pooled Data: {best_model_name} ===")
print(f"RMSE:     {np.sqrt(mean_squared_error(pareto_amp_holdout, holdout_pred)):.4f}")
print(f"MAE:      {mean_absolute_error(pareto_amp_holdout, holdout_pred):.4f}")
print(f"R^2:      {r2_score(pareto_amp_holdout, holdout_pred):.4f}")
print(f"Spearman: {spearmanr(pareto_amp_holdout, holdout_pred)[0]:.4f}")

### Best Model Selection and Final Evaluation on Full Train-/Test-/Holdout data

In [ ]:
best_model_name = None
best_r2 = -np.inf

for model_name, metrics in global_results_full.items():
    r2 = metrics["R2"][0]
    if r2 > best_r2:
        best_r2 = r2
        best_model_name = model_name

print(f"Best model: {best_model_name} (R^2 = {best_r2:.4f}) \n")

fit_function, predict_function = models[best_model_name]
final_model = fit_function(speed_train, amp_train)
holdout_pred = predict_function(final_model, speed_holdout)

print(f"=== Final Holdout Evaluation on Full Data: {best_model_name} ===")
print(f"RMSE:     {np.sqrt(mean_squared_error(amp_holdout, holdout_pred)):.4f}")
print(f"MAE:      {mean_absolute_error(amp_holdout, holdout_pred):.4f}")
print(f"R^2:      {r2_score(amp_holdout, holdout_pred):.4f}")
print(f"Spearman: {spearmanr(amp_holdout, holdout_pred)[0]:.4f}")